In [ ]:
%load_ext autoreload
%autoreload 2

# core.dom.summary.natural_text


>DOM natural text summary 
output-file: core.dom.summary.natural_text
title: core.dom.summary.natural_text

This notebook provides utility functions for DOM natural text summary
---

In [ ]:
# | default_exp core.dom.summary.natural_text

In [ ]:
# | export
from IPython.core.interactiveshell import InteractiveShell
from nbdev.showdoc import *

shell = InteractiveShell.instance()
shell.ast_node_interactivity = "all"

In [ ]:
#| export
from fastcore.test import *


In [ ]:
# | export
import asyncio
from contextvars import ContextVar
from functools import wraps
from inspect import signature
import json
import os
import re
import sys
from collections.abc import Callable
from pathlib import Path
from typing import Optional
from weakref import WeakKeyDictionary


project_root = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "pyproject.toml").exists()),
    None,
 )
if project_root is not None and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ollama import AsyncClient
try:
    from openai import AsyncOpenAI
except ModuleNotFoundError:
    class AsyncOpenAI:  # type: ignore[no-redef]
        pass

from ribosome.core.dom.utils import (
    get_summary_response_async, 
    get_image_summary_response_async, 
    get_text_summary_response_async,
    image_link_pattern,
    get_image_summary_async,
 )

from ribosome.core.dom.utils import with_async_context


class _LocatedSummaryError(ValueError):
    """An error already carrying the originating AST location."""


def _positive_setting(value, name):
    try:
        if isinstance(value, bool):
            raise ValueError
        parsed = int(value)
        if str(parsed) != str(value).strip() or parsed < 1:
            raise ValueError
    except (ValueError, TypeError):
        raise ValueError(f"{name} must be a positive integer; got {value!r}") from None
    return parsed


_SUMMARY_CONCURRENCY = _positive_setting(os.getenv('RIBOSOME_SUMMARY_CONCURRENCY', '8'), 'RIBOSOME_SUMMARY_CONCURRENCY')
_SUMMARY_TASK_LIMIT = _positive_setting(os.getenv('RIBOSOME_SUMMARY_TASK_LIMIT', '8'), 'RIBOSOME_SUMMARY_TASK_LIMIT')
_summary_semaphores = WeakKeyDictionary()
_summary_budget = ContextVar('summary_budget', default=None)


def _get_summary_semaphore():
    loop = asyncio.get_running_loop()
    limit = _positive_setting(_SUMMARY_CONCURRENCY, 'RIBOSOME_SUMMARY_CONCURRENCY')
    semaphore = _summary_semaphores.get(loop)
    if semaphore is None:
        semaphore = _summary_semaphores[loop] = asyncio.Semaphore(limit)
    return semaphore


def _with_summary_context(message):
    """Bind actual arguments before locating failures; cancellation passes through."""
    def decorate(fn):
        sig = signature(fn)

        @wraps(fn)
        async def wrapped(*args, **kwargs):
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()
            token = None
            if _summary_budget.get() is None:
                limit = _positive_setting(_SUMMARY_TASK_LIMIT, 'RIBOSOME_SUMMARY_TASK_LIMIT')
                _positive_setting(_SUMMARY_CONCURRENCY, 'RIBOSOME_SUMMARY_CONCURRENCY')
                token = _summary_budget.set({'limit': limit, 'active': 0})
            try:
                return await fn(*args, **kwargs)
            except _LocatedSummaryError:
                raise
            except Exception as exc:
                raise _node_error(f'{message}: {exc}', bound.arguments.get('path', '$'), bound.arguments['node']) from exc
            finally:
                if token is not None:
                    _summary_budget.reset(token)
        return wrapped
    return decorate


async def _ordered_summary_map(fn, values):
    """At most task_limit spawned workers per traversal, including nested maps.

    Single-item maps run inline without reserving a slot. Reserved slots stay
    with workers until completion. Nested maps run inline when
    all slots are occupied; they never wait for parents to release a task slot.
    Result storage is O(n); active tasks are bounded independently of AST width.
    """
    budget = _summary_budget.get()
    token = None
    if budget is None:
        limit = _positive_setting(_SUMMARY_TASK_LIMIT, 'RIBOSOME_SUMMARY_TASK_LIMIT')
        budget = {'limit': limit, 'active': 0}
        token = _summary_budget.set(budget)
    tasks = []
    try:
        if len(values) == 1:
            return [await fn(0, values[0])]
        results = [None] * len(values)
        count = min(len(values), budget['limit'] - budget['active'])
        if count == 0:
            for index, value in enumerate(values):
                results[index] = await fn(index, value)
            return results
        pending = iter(enumerate(values))

        async def worker():
            try:
                for index, value in pending:
                    results[index] = await fn(index, value)
            finally:
                budget['active'] -= 1

        for _ in range(count):
            # Reservation is synchronous before spawning, shared by descendants.
            budget['active'] += 1
            tasks.append(asyncio.create_task(worker()))
        try:
            await asyncio.gather(*tasks)
        except BaseException:
            for task in tasks:
                task.cancel()
            await asyncio.gather(*tasks, return_exceptions=True)
            raise
        return results
    finally:
        if token is not None:
            _summary_budget.reset(token)


def _summary_child_path(path, key):
    if isinstance(key, int):
        return f'{path}[{key}]'
    if re.fullmatch(r'[A-Za-z_][A-Za-z_0-9]*', key):
        return f'{path}.{key}'
    return f'{path}[{json.dumps(key, ensure_ascii=False)}]'


def _apply_summary_action(node, action, path):
    """Once per traversed value; replacements preserve dict/list/scalar category."""
    transformed = action(node) if action is not None else node
    for category in (dict, list):
        if isinstance(node, category) != isinstance(transformed, category):
            raise _node_error('Action must preserve node container category', path, node)
    if not isinstance(transformed, (dict, list, str, int, float, bool, type(None))):
        raise _node_error(f'Unsupported node type: {type(transformed)}', path, transformed)
    return transformed


In [ ]:
# | hide
import asyncio
import io
import threading
from contextlib import redirect_stdout
from tempfile import TemporaryDirectory
from unittest.mock import patch

In [ ]:
#| export
from dotenv import load_dotenv

load_dotenv()
OLLAMA_API_KEY= os.getenv('OLLAMA_API_KEY')
DASHSCOPE_API_KEY= os.getenv('DASHSCOPE_API_KEY')

In [ ]:
# | hide
def run_async(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result = {}
    error = {}

    def _runner():
        try:
            result['value'] = asyncio.run(coro)
        except BaseException as exc:
            error['value'] = exc

    thread = threading.Thread(target=_runner, daemon=True)
    thread.start()
    thread.join()
    if 'value' in error:
        raise error['value']
    return result.get('value')


In [ ]:
# | export
async def get_leaf_summary_async(node:str, min_len:int, client: AsyncClient | AsyncOpenAI, model:str) -> str: # If the string length is less than 200, return the node as is
    """ Generate a summary for a given text node.

    Args:
        node (str): The text node to summarize.
        min_len (int): The minimum length of the text node to summarize.

    Returns:
        str: The summary of the text node.
    """
    
    if len(node) < min_len:
        return node
    # If the node is not an image link, return a summary of the text using unified function
    async with _get_summary_semaphore():
        response_content = await get_summary_response_async(client, node, model=model, role="user", lang='zh')
    if response_content:
        return response_content
    else:
        # If the response is empty, return the original node
        return node


In [ ]:
# | hide
def test_get_leaf_summary_async_threshold_and_fallback():
    async def fake_summary(client, content, model, role, lang):
        return f'SUM<{content}>'

    with patch.dict(globals(), {'get_summary_response_async': fake_summary}):
        test_eq(run_async(get_leaf_summary_async('hi', 5, object(), 'm')), 'hi')
        test_eq(run_async(get_leaf_summary_async('hello', 3, object(), 'm')), 'SUM<hello>')

    async def empty_summary(client, content, model, role, lang):
        return ''

    with patch.dict(globals(), {'get_summary_response_async': empty_summary}):
        test_eq(run_async(get_leaf_summary_async('hello', 3, object(), 'm')), 'hello')


test_get_leaf_summary_async_threshold_and_fallback()

In [ ]:
# | export
async def get_list_summary_async(root: list, leaf_min_len: int, client: AsyncClient | AsyncOpenAI, model:str) -> str:
    """
    Given a list of strings, return a summary of the list.
    If the list is empty, return an empty string.
    If the list has only one element, return the summary of that element.
    If the list has more than one element, return a summary of the concatenated elements.
    """
    list_summary = []
    for n in root:
        if isinstance(n, str):
            # If the element is a string, get its summary
            summary = await get_leaf_summary_async(n, min_len=leaf_min_len, client=client, model=model)
        elif isinstance(n, int) or isinstance(n, float):
            # If the element is a number, get its summary
            summary = await get_leaf_summary_async(str(n), min_len=leaf_min_len, client=client, model=model)
        elif isinstance(n, dict):
            # If the dict has a 's' key, use it as the summary
            summary = n.get('s', '')
        elif isinstance(n, list):
            # If the element is a list, get its summary
            summary = await get_list_summary_async(n, leaf_min_len=leaf_min_len, client=client, model=model)
        elif n is None:
            # If the element is None, skip it
            summary = ""
        else:
            raise ValueError(f"Unsupported element type: {type(n)} in {n}")
        list_summary.append(summary)
        
    # Concatenate all elements and summarize
    # concatenated = " ".join(list_summary)
    # response = get_text_summary_response(concatenated, model="gemma3:27b", role="user", lang='zh')
    # if response.message.content:
    #     return response.message.content
    # else:
    #     # If the response is empty, raise an exception
    #     raise ValueError("Summary response is empty. Please check the input data.")
    return await get_leaf_summary_async(" ".join(list_summary), min_len=leaf_min_len, client=client, model=model)


In [ ]:
# | hide
def test_get_list_summary_async_handles_supported_values():
    async def identity_leaf(node, min_len, client, model):
        return node

    with patch.dict(globals(), {'get_leaf_summary_async': identity_leaf}):
        result = run_async(get_list_summary_async(['a', 2, {'s': 'dict'}, ['nested'], None], 0, object(), 'm'))

    test_eq(result.strip(), 'a 2 dict nested')
    test_fail(lambda: run_async(get_list_summary_async([('bad',)], 0, object(), 'm')), contains='Unsupported element type')


test_get_list_summary_async_handles_supported_values()

In [ ]:
# | export
def _node_error(message: str, path: str, node=None) -> ValueError:
    """Build a ValueError locating a failure by node type and key path within the AST."""
    node_type = node.get("t") if isinstance(node, dict) else type(node).__name__
    details = [message, f"Node type: {node_type}", f"At path: {path}"]
    if node is not None:
        details.insert(1, f"Node: {node}")
    return _LocatedSummaryError("\n".join(details))


def _resolve_image_link(node: dict, root_path: Path, path: str) -> str:
    """Resolve the link of an image node."""
    try:
        image_path = Path(node['c'][2][0])
    except (IndexError, KeyError, TypeError) as exc:
        raise _node_error('Invalid image node structure', path, node) from exc

    if not str(image_path) or not re.search(image_link_pattern, str(image_path)):
        raise _node_error(f'Invalid image link: {image_path}', path, node)
    return str((root_path / image_path).resolve())


In [ ]:
# | hide
def test_resolve_image_link_validates_shape_and_extension():
    root = Path('/tmp/root')
    test_eq(
        _resolve_image_link({'c': [None, None, ['robot.png', '']]}, root, '$'),
        str(root / 'robot.png'),
    )
    test_eq(
        _resolve_image_link({'c': [None, None, ['images/../robot.png', '']]}, Path('docs'), '$'),
        str((Path('docs') / 'robot.png').resolve()),
    )
    test_fail(
        lambda: _resolve_image_link({'c': []}, root, '$.blocks[0].c[1]'),
        contains='Invalid image node structure',
    )
    test_fail(
        lambda: _resolve_image_link({'c': [None, None, ['robot.txt', '']]}, root, '$.blocks[0].c[1]'),
        contains='Invalid image link',
    )
    test_fail(
        lambda: _resolve_image_link({'c': []}, root, '$.blocks[0].c[1]'),
        contains='At path: $.blocks[0].c[1]',
    )


test_resolve_image_link_validates_shape_and_extension()


In [ ]:
# | export
@_with_summary_context('Error summarizing image node')
async def _summarize_image_node_async(node: dict, root_path: Path, client: AsyncClient|AsyncOpenAI, model: str, path: str = "$", lang: str = "zh") -> dict:
    """Summarize an image node asynchronously.

    Args:
        node (dict): The image node.
        root_path (Path): Document root used to resolve the relative image link.
        client (AsyncClient|AsyncOpenAI): Multimodal client used for the image summary.
        model (str): Model name passed to the client.
        path (str): Key path of the node within the AST, used for logging and errors.
        lang (str): Language forwarded to the image-summary provider.

    Returns:
        dict: The summarized image node.
    """
    if not isinstance(node.get('c', []), list):
        raise _node_error("Invalid image caption format", path, node)

    image_link = _resolve_image_link(node, root_path, path)
    async with _get_summary_semaphore():
        node["s"] = await get_image_summary_async(
            client=client,
            image_link=image_link,
            model=model,
            role="user",
            lang=lang
        )
    print(f"Summarize image: {image_link} at {path}")
    return node


In [ ]:
# | hide
def test_summarize_image_node_async_sets_summary_and_validates_caption():
    async def fake_image_summary(client, image_link, model, role, lang):
        return f'IMG<{Path(image_link).name}>'

    with TemporaryDirectory() as tmpdir:
        root = Path(tmpdir)
        node = {'t': 'Image', 'c': [['', [], []], [], ['robot.png', '']]}
        stdout = io.StringIO()
        with patch.dict(globals(), {'get_image_summary_async': fake_image_summary}):
            with redirect_stdout(stdout):
                result = run_async(_summarize_image_node_async(node, root, object(), 'm', path='$.blocks[0].c[1]'))

    test_eq(result['s'], 'IMG<robot.png>')
    test_eq('Summarize image:' in stdout.getvalue(), True)
    test_eq('at $.blocks[0].c[1]' in stdout.getvalue(), True)
    test_fail(lambda: run_async(_summarize_image_node_async({'t': 'Image', 'c': 'bad'}, Path('.'), object(), 'm')), contains='Invalid image caption format')


test_summarize_image_node_async_sets_summary_and_validates_caption()


In [ ]:
# | export
async def _summarize_leaf_async(node: str, leaf_min_len: int, min_len: Optional[int], client: AsyncClient|AsyncOpenAI, model: str, lang: str) -> str:
    """Summarize a leaf node asynchronously."""
    text = str(node).strip()
    threshold = leaf_min_len if min_len is None else min_len
    if len(text) < threshold:
        return text

    async with _get_summary_semaphore():
        response_content = await get_summary_response_async(
            client,
            text,
            model=model,
            role='user',
            lang=lang,
        )
    return response_content or text


In [ ]:
# | hide
def test_summarize_leaf_async_respects_threshold_and_fallback():
    async def fake_summary(client, text, model, role, lang):
        return f'TXT<{text}>'

    with patch.dict(globals(), {'get_summary_response_async': fake_summary}):
        test_eq(run_async(_summarize_leaf_async('hi', 5, None, object(), 'm', 'zh')), 'hi')
        test_eq(run_async(_summarize_leaf_async('hello', 1, None, object(), 'm', 'zh')), 'TXT<hello>')

    async def empty_summary(client, text, model, role, lang):
        return ''

    with patch.dict(globals(), {'get_summary_response_async': empty_summary}):
        test_eq(run_async(_summarize_leaf_async('hello', 1, None, object(), 'm', 'zh')), 'hello')


test_summarize_leaf_async_respects_threshold_and_fallback()

In [ ]:
# | export
async def _coerce_summary_async(value: str|int|float|dict|list|None, leaf_min_len: int, min_len: Optional[int], client: AsyncClient|AsyncOpenAI, model: str, lang: str) -> str:
    """Coerce a value into a summarized string.

    Args:
        self (DOMClass): The DOMClass instance.
        value: The value to be coerced into a summarized string.

    Raises:
        ValueError: If the value type is unsupported.

    Returns:
        str: The summarized string.
    """
    if isinstance(value, str):
        return await _summarize_leaf_async(value, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)
    if isinstance(value, (int, float)):
        return await _summarize_leaf_async(str(value), leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)
    if isinstance(value, dict):
        return value.get('s', '')
    if isinstance(value, list):
        return await _summarize_list_async(value, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)
    if value is None:
        return ""
    raise ValueError(f"Unsupported element type: {type(value)} in {value}")


In [ ]:
# | hide
def test_coerce_summary_async_handles_supported_shapes():
    async def fake_leaf(node, leaf_min_len, min_len, client, model, lang):
        return f'LEAF<{node}>'

    async def fake_list(value, leaf_min_len, min_len, client, model, lang):
        return f'LIST<{len(value)}>'

    with patch.dict(globals(), {'_summarize_leaf_async': fake_leaf, '_summarize_list_async': fake_list}):
        test_eq(run_async(_coerce_summary_async('x', 1, 2, object(), 'm', 'zh')), 'LEAF<x>')
        test_eq(run_async(_coerce_summary_async(2, 1, 2, object(), 'm', 'zh')), 'LEAF<2>')
        test_eq(run_async(_coerce_summary_async({'s': 'dict'}, 1, 2, object(), 'm', 'zh')), 'dict')
        test_eq(run_async(_coerce_summary_async(['a', 'b'], 1, 2, object(), 'm', 'zh')), 'LIST<2>')
        test_eq(run_async(_coerce_summary_async(None, 1, 2, object(), 'm', 'zh')), '')

    test_fail(lambda: run_async(_coerce_summary_async({1, 2}, 1, 2, object(), 'm', 'zh')), contains='Unsupported element type')


In [ ]:
# | export
async def _summarize_list_async(
    root: list,
    leaf_min_len: int,
    min_len: Optional[int],
    client: AsyncClient|AsyncOpenAI,
    model: str,
    lang: str
) -> str:
    """Summarize a list of values asynchronously."""
    async def coerce_item(index, item):
        return await _coerce_summary_async(item, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)
    parts = await _ordered_summary_map(coerce_item, root)
    merged = ' '.join(part for part in parts if part)
    return await _summarize_leaf_async(merged, leaf_min_len=leaf_min_len, min_len=min_len, client=client, model=model, lang=lang)


In [ ]:
# | hide
def test_summarize_list_async_merges_nonempty_parts():
    async def fake_coerce(item, leaf_min_len, min_len, client, model, lang):
        return {'a': 'A', 'b': '', 'c': 'C'}[item]

    async def fake_leaf(node, leaf_min_len, min_len, client, model, lang):
        return f'SUM<{node}>'

    with patch.dict(globals(), {'_coerce_summary_async': fake_coerce, '_summarize_leaf_async': fake_leaf}):
        result = run_async(_summarize_list_async(['a', 'b', 'c'], 0, None, object(), 'm', 'zh'))

    test_eq(result, 'SUM<A C>')


test_coerce_summary_async_handles_supported_shapes()
test_summarize_list_async_merges_nonempty_parts()

In [ ]:
# | export
def _record_summary_progress(node_type: str, path: str, file_path: Path, table_count: dict, section_count: dict) -> None:
    """Record the progress of summarizing a node.

    Args:
        node_type (str): The type of the node being summarized.
        path (str): Key path of the node within the AST.
        file_path (Path): Path of the document being summarized.
        table_count (dict): Mutable cumulative table counter ({"count": int}), incremented in place.
        section_count (dict): Mutable cumulative section counter ({"count": int}), incremented in place.
    """
    if node_type == "Table":
        table_count["count"] += 1
        print(f"{file_path} Summarize table: {table_count['count']} at {path}")
    elif node_type == "Section":
        section_count["count"] += 1
        print(f"{file_path} Summarize section: {section_count['count']} at {path}")


In [ ]:
# | hide
def test_record_summary_progress_prints_expected_labels():
    stdout = io.StringIO()
    table_counter = {"count": 1}
    section_counter = {"count": 2}
    with redirect_stdout(stdout):
        _record_summary_progress('Table', '$.blocks[0].c[1]', Path('doc.md'), table_counter, section_counter)
        _record_summary_progress('Section', '$.blocks[0]', Path('doc.md'), table_counter, section_counter)

    output = stdout.getvalue()
    test_eq('Summarize table: 2 at $.blocks[0].c[1]' in output, True)
    test_eq('Summarize section: 3 at $.blocks[0]' in output, True)
    test_eq(table_counter["count"], 2)
    test_eq(section_counter["count"], 3)


test_record_summary_progress_prints_expected_labels()


In [ ]:
# | export
@_with_summary_context('Error summarizing node')
async def summarize_node_async(
    node: dict | list | str | int | float | bool | None,
    root_path: Path,
    client: AsyncClient | AsyncOpenAI,
    model: str,
    file_path: Path,
    table_count: dict,
    section_count: dict,
    path: str = '$',
    action: Optional[Callable] = None,
    leaf_min_len: int = 100,
    min_len: Optional[int] = None,
    lang: str = 'zh',
) -> dict | list | str | int | float | bool | None:
    """Walk one AST with bounded ordered work and awaited sibling cleanup.

    Actions run once on traversed containers/scalars except t/s and image/ignored
    node internals. Replacements preserve container category. Callbacks must avoid
    shared order-dependent state. Scalar coercion retains the legacy text contract.
    """
    node = _apply_summary_action(node, action, path)
    if isinstance(node, dict):
        node_type = node.get('t')
        if node_type is None:
            raise _node_error("Node does not have a 't' key", path, node)
        if node_type == 'Image' and not node.get('c'):
            node['s'] = ''
            return node
        if node_type == 'Image':
            return await _summarize_image_node_async(node, root_path, client, model, path, lang)
        if node_type in {'Cite', 'AlignDefault', 'ColWidth'}:
            node['s'] = ''
            return node
        parts = []
        for key, value in list(node.items()):
            if key in {'t', 's'}:
                continue
            child_path = _summary_child_path(path, key)
            if isinstance(value, list):
                items = _apply_summary_action(value, action, child_path)

                async def summarize_item(index, item):
                    return await summarize_node_async(
                        item, root_path, client, model, file_path, table_count, section_count,
                        _summary_child_path(child_path, index), action, leaf_min_len, min_len, lang)

                children = await _ordered_summary_map(summarize_item, items)
                node[key] = children
                if children:
                    parts.append(await _summarize_list_async(children, leaf_min_len, min_len, client, model, lang))
            elif isinstance(value, dict):
                child = await summarize_node_async(
                    value, root_path, client, model, file_path, table_count, section_count,
                    child_path, action, leaf_min_len, min_len, lang)
                node[key] = child
                parts.append(child.get('s', ''))
            else:
                value = _apply_summary_action(value, action, child_path)
                node[key] = value
                if value is not None and value != '':
                    parts.append(await _summarize_leaf_async(str(value), leaf_min_len, min_len, client, model, lang))
        merged = ' '.join(part for part in parts if part)
        node['s'] = await _summarize_leaf_async(merged, leaf_min_len, min_len, client, model, lang) if node.get('c') and merged else ''
        _record_summary_progress(node_type, path, file_path, table_count, section_count)
    elif isinstance(node, list):
        async def summarize_item(index, item):
            item_path = _summary_child_path(path, index)
            if isinstance(item, (dict, list)):
                return await summarize_node_async(
                    item, root_path, client, model, file_path, table_count, section_count,
                    item_path, action, leaf_min_len, min_len, lang)
            item = _apply_summary_action(item, action, item_path)
            return str(item).strip() if item is not None else None
        node = await _ordered_summary_map(summarize_item, node)
    elif node is not None:
        node = await _summarize_leaf_async(str(node), leaf_min_len, min_len, client, model, lang)
    return node


In [ ]:
# | hide
def test_summarize_node_async_summarizes_dict_list_and_scalar_paths():
    payload = {'t': 'Section', 'c': [{'t': 'Header', 'c': [2]}, 'hello', {'t': 'Cite', 'c': []}]}
    progress = []

    async def fake_leaf(node, leaf_min_len, min_len, client, model, lang):
        return f'S<{node}>'

    async def fake_list(root, leaf_min_len, min_len, client, model, lang):
        return f'L<{len(root)}>'

    def fake_progress(node_type, path, file_path, table_count, section_count):
        progress.append((node_type, path, table_count["count"], section_count["count"]))

    with patch.dict(globals(), {'_summarize_leaf_async': fake_leaf, '_summarize_list_async': fake_list, '_record_summary_progress': fake_progress}):
        result = run_async(
            summarize_node_async(json.loads(json.dumps(payload)), Path('.'), client=object(), model='m', file_path=Path('doc.md'), table_count={"count": 1}, section_count={"count": 2}, leaf_min_len=0, min_len=None)
        )

    test_eq(result['c'][0]['s'], 'S<L<1>>')
    test_eq(result['c'][1], 'S<hello>')
    test_eq(result['c'][2]['s'], '')
    test_eq(result['s'], 'S<L<3>>')
    test_eq(progress, [('Header', '$.c[0]', 1, 2), ('Section', '$', 1, 2)])
    test_fail(lambda: run_async(summarize_node_async({}, Path('.'), object(), 'm', Path('doc.md'), {"count": 0}, {"count": 0})), contains="Node does not have a 't' key")


def test_summarize_node_async_reports_key_path_in_errors():
    payload = {'t': 'Section', 'c': [{'c': []}]}  # the child has no 't' key

    with patch.dict(globals(), {'_record_summary_progress': lambda *a, **k: None}):
        test_fail(
            lambda: run_async(summarize_node_async(payload, Path('.'), object(), 'm', Path('doc.md'), {"count": 0}, {"count": 0}, path='$.blocks[0]')),
            contains='At path: $.blocks[0].c[0]',
        )


def test_summarize_node_async_default_leaf_min_len_avoids_llm_calls():
    calls = []

    async def counting_summary(client, text, model, role, lang):
        calls.append(text)
        return 'LLM'

    with patch.dict(globals(), {'get_summary_response_async': counting_summary}):
        result = run_async(
            summarize_node_async('short text', Path('.'), client=object(), model='m', file_path=Path('doc.md'), table_count={"count": 0}, section_count={"count": 0})
        )

    test_eq(result, 'short text')
    test_eq(calls, [])


def test_summarize_node_async_gathers_siblings_concurrently():
    payload = {'t': 'Section', 'c': [
        {'t': 'Para', 'c': ['alpha']},
        {'t': 'Para', 'c': ['beta']},
        {'t': 'Para', 'c': ['gamma']},
    ]}

    async def scenario():
        saturated, release = asyncio.Event(), asyncio.Event()
        inflight = {'now': 0, 'max': 0}
        async def leaf(node, *args, **kwargs):
            inflight['now'] += 1
            inflight['max'] = max(inflight['max'], inflight['now'])
            if inflight['now'] >= 2:
                saturated.set()
            try:
                await release.wait()
                return f'S<{node}>'
            finally:
                inflight['now'] -= 1
        async def fake_list(root, *args, **kwargs):
            return f'L<{len(root)}>'
        with patch.dict(globals(), {'_summarize_leaf_async': leaf, '_summarize_list_async': fake_list,
                                    '_SUMMARY_TASK_LIMIT': 3, '_record_summary_progress': lambda *a, **k: None}):
            task = asyncio.create_task(summarize_node_async(
                payload, Path('.'), object(), 'm', Path('doc.md'), {'count': 0}, {'count': 0}))
            try:
                await asyncio.wait_for(saturated.wait(), 2)
                release.set()
                result = await asyncio.wait_for(task, 2)
            finally:
                release.set()
                task.cancel()
                await asyncio.gather(task, return_exceptions=True)
        test_eq(inflight['max'] > 1, True)
        test_eq([child['c'][0] for child in result['c']], ['S<alpha>', 'S<beta>', 'S<gamma>'])
        test_eq([child['s'] for child in result['c']], ['S<L<1>>'] * 3)
        test_eq(result['s'], 'S<L<3>>')
    run_async(scenario())


def test_summarize_leaf_async_caps_llm_fanout_with_semaphore():
    async def scenario():
        saturated, release = asyncio.Event(), asyncio.Event()
        inflight = {'now': 0, 'max': 0}
        async def tracking_summary(client, text, **kwargs):
            inflight['now'] += 1
            inflight['max'] = max(inflight['max'], inflight['now'])
            if inflight['now'] == _SUMMARY_CONCURRENCY:
                saturated.set()
            try:
                await release.wait()
                return f'TXT<{text}>'
            finally:
                inflight['now'] -= 1
        async def fan_out():
            return await asyncio.gather(*(
                _summarize_leaf_async('long enough text', 1, None, object(), 'm', 'zh')
                for _ in range(_SUMMARY_CONCURRENCY * 3)))
        with patch.dict(globals(), {'get_summary_response_async': tracking_summary}):
            task = asyncio.create_task(fan_out())
            try:
                await asyncio.wait_for(saturated.wait(), 2)
                release.set()
                results = await asyncio.wait_for(task, 2)
            finally:
                release.set()
                task.cancel()
                await asyncio.gather(task, return_exceptions=True)
        test_eq(len(results), _SUMMARY_CONCURRENCY * 3)
        test_eq(results[0], 'TXT<long enough text>')
        test_eq(inflight['max'], _SUMMARY_CONCURRENCY)
    run_async(scenario())


def test_summarize_node_async_threads_action_through_every_depth():
    payload = {'t': 'Section', 'c': [{'t': 'Para', 'c': ['alpha'], 'subnodes': {'t': 'Str', 'c': 'x'}}]}
    visited = []

    def tag(node):
        visited.append(node.get('t') if isinstance(node, dict) else type(node).__name__)
        return node

    async def fake_summary(client, text, model, role, lang):
        return f'[{text}]'

    with patch.dict(globals(), {'get_summary_response_async': fake_summary, '_record_summary_progress': lambda *a, **k: None}):
        run_async(
            summarize_node_async(payload, Path('.'), client=object(), model='m', file_path=Path('doc.md'), table_count={"count": 0}, section_count={"count": 0}, action=tag, leaf_min_len=0, min_len=0)
        )

    test_eq(visited, ['Section', 'list', 'Para', 'list', 'str', 'Str', 'str'])


test_summarize_node_async_summarizes_dict_list_and_scalar_paths()
test_summarize_node_async_reports_key_path_in_errors()
test_summarize_node_async_default_leaf_min_len_avoids_llm_calls()
test_summarize_node_async_gathers_siblings_concurrently()
test_summarize_leaf_async_caps_llm_fanout_with_semaphore()
test_summarize_node_async_threads_action_through_every_depth()


In [ ]:
# | export
async def summary_node_async(
        node: dict | list,
        root_path: Path,
        file_path: Path,
        client: AsyncClient|AsyncOpenAI,
        model: str,
        table_count: int,
        section_count: int,
        action: Optional[Callable] = None,
        leaf_min_len: int = 0,
        min_len: Optional[int] = None,
        lang: str='zh',
    ) -> dict | list:
    '''
    Given a string node, add key,value pair: node['s'] = node_summary, and return the node 
    '''

    if isinstance(node, dict):
        try:
            t = node["t"]
        except KeyError:
            raise ValueError(f"Node does not have a 't' key: {node}")
        if t == "Image":  # Image summary, the Image node is as defined in the pandoc AST
            summary = []
            if (not node['c'][1] == []) and (node['c'][1][0] is not None) and (node['c'][1][0].get('c') is not None):
                summary.append(node['c'][1][0]['c'])  # The content of the second element is the image caption
            try:
                # summary.append(node['c'][0])  # The first element are defined to be attributes of the image rendering, i.e. content-irrelevant.
                # If the node is an image, get its link
                image_link = root_path / node['c'][2][0]  # Assuming the image link is in the third element of the list
                image_link = str(image_link)
                if image_link and re.search(image_link_pattern, image_link):  # re.match leads to empty match if there's spaces in the path!
                    # If the node is an image link, get its summary
                    assert isinstance(client,AsyncClient), "Only ollama client is allowed for image summary"
                    async with _get_summary_semaphore():
                        response = await get_image_summary_response_async(
                            client=client,
                            image_link=image_link,
                            model=model,
                            role="user",
                            lang=lang
                            )
                    summary.append(response.message.content)
                else:
                    # If the node is not an image link, summarize its content
                    raise ValueError(f"Invalid image link: {image_link}")
                # Get the summary of the image caption
            except (IndexError, KeyError):
                # Handle cases where the image link is not in the expected format
                raise ValueError(f"Invalid image node structure: {node}")
            # The second element is the image title
            summary.append(node['c'][2][1])  # The second element is the image title
            assert isinstance(client,AsyncClient), "Only ollama client is allowed for image summary"
            async with _get_summary_semaphore():
                response_txt = await get_text_summary_response_async(
                    client=client,
                    content=" ".join(summary),
                    model=model,
                    role="user",
                    lang=lang
                )
            node["s"] = response_txt.message.content
            print(f"Summarize image: {image_link}")
        elif t == "Cite" or t == "AlignDefault" or t == "ColWidth" :  # Quoted node will be ignored
            node["s"] = ""  # Set the summary to an empty string
        else: # TextBlock summary
            c = node.get("c", None)
            if not c:
                node["s"] = ""
            else:
                dict_summary = []
                for key, value in node.items(): # get summary of the values (content)
                    if key == "t":
                        continue
                    if isinstance(value, list):  # get the summary of the string list
                        if value == []:
                            # If the list is empty, skip it
                            continue
                        # If the value is a list, summarize each element
                        node[key] = [
                            await summary_node_async(
                                node=child,
                                root_path=root_path,
                                file_path=file_path,
                                client=client,
                                model=model,
                                table_count=table_count,
                                section_count=section_count,
                                action=action,
                                leaf_min_len=leaf_min_len,
                                min_len=min_len,
                                lang=lang
                                )
                            if isinstance(child, (dict,list))
                            else child
                            for child in value
                        ]
                        dict_summary.append(
                            await _summarize_list_async(
                                root=value,
                                leaf_min_len=leaf_min_len,
                                min_len=min_len,
                                client=client,
                                model=model,
                                lang=lang)
                            )
                    elif isinstance(value, dict):
                        child = await summary_node_async(
                            node=value,
                            root_path=root_path,
                            file_path=file_path,
                            client=client,
                            model=model,
                            table_count=table_count,
                            section_count=section_count,
                            action=action,
                            leaf_min_len=leaf_min_len,
                            min_len=min_len,
                            lang=lang
                        )  # insert the value['s']
                        assert isinstance(child, dict) and 's' in child, f"Expected dict with 's' key, got {child}"
                        dict_summary.append(child['s'])
                    elif value is None or value == "":
                        # If the value is None, skip it
                        continue
                    else:
                        if not isinstance(value, str):
                            # If the value is not a string, convert it to a string
                            value = str(value)
                        dict_summary.append(
                            await _summarize_leaf_async(
                                node=value,
                                leaf_min_len=leaf_min_len,
                                min_len=min_len,
                                client=client,
                                model=model,
                                lang=lang
                                )
                            )
                # get the summary of the node
                if dict_summary:  # type: ignore
                    # If there are summaries, concatenate them
                    node["s"] = await _summarize_leaf_async(
                        node=" ".join(dict_summary),
                        leaf_min_len=leaf_min_len,
                        min_len=min_len,
                        client=client,
                        model=model,
                        lang=lang
                    )
                    # node["s"] = get_text_summary_response(
                    #     " ".join(dict_summary), model="gemma3:27b", role="user", lang='zh'
                    # ).message.content
                else:
                    # If no summaries, set to empty string
                    node["s"] = ""
            # if t is table
            if t == "Table":
                print(f"{file_path} Summarize table: {table_count}")
                table_count += 1
            elif t == "Section":
                print(f"{file_path} Summarize section: {section_count} section depth {node['c'][0]['c'][0]}")
                section_count += 1
    elif isinstance(node, list):
        node = [
            await summary_node_async(
                node=child,
                root_path=root_path,
                file_path=file_path,
                client=client,
                model=model,
                table_count=table_count,
                section_count=section_count,
                action=action,
                leaf_min_len=leaf_min_len,
                min_len=min_len,
                lang=lang
                ) if isinstance(child, (dict,list)) else child
            for child in node
        ]
    else:
        assert isinstance(node, str), f"Expected a string node, got {type(node)}, on json line"
        # If the node is a string, get its summary
        node_summary = await _summarize_leaf_async(
            node=node,
            leaf_min_len=leaf_min_len,
            min_len=min_len,
            client=client,
            model=model,
            lang=lang
        )
        node = node_summary
    return node

In [ ]:
# | hide
def test_summary_node_async_summarizes_legacy_node_shapes():
    payload = {'t': 'Section', 'c': [{'t': 'Header', 'c': [2]}, 'hello', {'t': 'Cite', 'c': []}]}

    async def fake_leaf(node, leaf_min_len, min_len, client, model, lang):
        return f'S<{node}>'

    async def fake_list(root, leaf_min_len, min_len, client, model, lang):
        return 'LIST'

    with patch.dict(globals(), {'_summarize_leaf_async': fake_leaf, '_summarize_list_async': fake_list}):
        result = run_async(summary_node_async(json.loads(json.dumps(payload)), Path('.'), Path('doc.md'), object(), 'm', 1, 2))

    test_eq(result['s'], 'S<LIST>')
    test_eq(result['c'][0]['s'], 'S<LIST>')
    test_eq(result['c'][1], 'hello')
    test_eq(result['c'][2]['s'], '')
    test_fail(lambda: run_async(summary_node_async({}, Path('.'), Path('doc.md'), object(), 'm', 0, 0)), contains="Node does not have a 't' key")


test_summary_node_async_summarizes_legacy_node_shapes()

# DOM Summary Functional Tests

In [ ]:
# | hide
# | notest
from ribosome.core.dom.model import DOMClass
from ribosome.system.utils import get_project_root

def _fresh_function_test_dom(asset_name):
    """Remove derived caches before setup so each manual test starts fresh."""
    project_root = get_project_root()
    md_file = ((project_root / 'assets' / asset_name / f'{asset_name}.md')
               if project_root is not None else Path('assets') / asset_name / f'{asset_name}.md')
    if not md_file.exists():
        md_file = Path('../assets') / asset_name / f'{asset_name}.md'
    assert md_file.is_file(), f'Markdown source missing: {md_file}'
    # Clear every registered artifact, including downstream semantics/embedding
    # caches that would otherwise describe the previous AST.
    for suffix in DOMClass._ARTIFACT_SUFFIXES.values():
        artifact = md_file.with_name(f'{md_file.stem}{suffix}')
        artifact.unlink(missing_ok=True)
    dom = DOMClass(md_file)
    dom.setup()
    assert dom.ast_json_file and dom.ast_json_file.exists(), 'AST was not regenerated'
    assert dom.ast_json and json.loads(dom.ast_json).get('blocks'), 'AST blocks missing'
    assert not dom.semantics_json and not dom.embed_json, 'Cached downstream artifacts loaded'
    print(f'Regenerated AST: {dom.ast_json_file}')
    return dom


## Summary

In [ ]:
# | hide
# | notest
async def test_textualize_with_sn024002_md():
    dom = _fresh_function_test_dom("SN024002")
    ast_before = dom.ast_json_file.read_bytes()
    print('Starting textualization...')
    await dom.summary_nodewline_main()
    assert dom.semantics_json and dom.semantics_json_file, 'Semantics were not generated'
    # Always replace the semantics artifact, regardless of previous runs.
    dom.semantics_json_file.write_text(dom.semantics_json, encoding='utf-8')
    test_eq(json.loads(dom.semantics_json_file.read_text(encoding='utf-8')),
            json.loads(dom.semantics_json))
    test_eq(dom.ast_json_file.read_bytes(), ast_before)
    print(f'Regenerated semantics: {dom.semantics_json_file}')


await test_textualize_with_sn024002_md()


In [ ]:
# | hide
# | notest
async def test_textualize_with_sx022001_md():
    dom = _fresh_function_test_dom("SX022001")
    ast_before = dom.ast_json_file.read_bytes()
    print('Starting textualization...')
    await dom.summary_nodewline_main()
    assert dom.semantics_json and dom.semantics_json_file, 'Semantics were not generated'
    # Always replace the semantics artifact, regardless of previous runs.
    dom.semantics_json_file.write_text(dom.semantics_json, encoding='utf-8')
    test_eq(json.loads(dom.semantics_json_file.read_text(encoding='utf-8')),
            json.loads(dom.semantics_json))
    test_eq(dom.ast_json_file.read_bytes(), ast_before)
    print(f'Regenerated semantics: {dom.semantics_json_file}')


await test_textualize_with_sx022001_md()


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()